# 01 — HAR-RV (weekly silver volatility)

First two models on the volatility ladder:

- **Naïve** — $\hat{\text{RV}}_t = \text{RV}_{t-1}$. The floor. Volatility has a strong
  AR(1) component, so last week's RV is already a hard baseline to beat.
- **HAR-RV** (Corsi 2009) — OLS regression of RV on its short / medium / long trailing
  averages.

Features come from `volatility_weekly.csv`, built in `00_features.ipynb` — run that
notebook first.


## Setup


In [8]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from vol_utils import vol_evaluate, vol_period_metrics, vol_diebold_mariano
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')

# HAR-RV is plain OLS with no hyperparameters -> train and val are pooled into `trval`
# for the final fit; we do not need separate `train_df` / `val_df` here. (The RF and
# XGB notebooks in `03`/`04` keep them separate because they grid-search hyperparameters
# on val; for HAR there is nothing to tune.)
test_df  = frame[frame['split'] == 'test']
trval_df = frame[frame['split'] != 'test']

FEATS_HAR = ['rv_w_lag1', 'rv_m_lag1', 'rv_q_lag1']
print(f'train+val: {len(trval_df)}   test: {len(test_df)}')

train+val: 405   test: 175


## 1. Naïve baseline — $\hat{\text{RV}}_t = \text{RV}_{t-1}$

The naïve prediction is simply last week's RV, which is exactly the `rv_w_lag1` column.
Because volatility is highly persistent this is a genuinely strong baseline — every
other model has to beat it to justify itself.

(Its DCA is ≈ 0 by construction: predicting last week's RV implies *no* change, so the
naïve model can never call a direction. It is a floor for RMSE/MAE, not for DCA.)


In [9]:
y_test    = test_df['target'].values
prev_test = test_df['rv_w_lag1'].values        # RV_{t-1}

results = []
results.append(vol_evaluate('Naive (RV_t-1)', y_test, prev_test, prev_test))


Naive (RV_t-1)                  RMSE=0.03979  MAE=0.02013  R2=-0.095  DCA=0.000


## 2. HAR-RV (Corsi 2009)

**HAR-RV** stands for the *Heterogeneous Autoregressive model of Realised Volatility*.
It is a simple linear regression that predicts realised volatility from its own past
values measured over several horizons. The "heterogeneous" in the name refers to the
economic story behind it: different market participants operate on different time
horizons — short-term traders care about last week, medium-term traders about last
month, longer-term traders about last quarter — so the volatility observed today
reflects a mixture of all of those horizons.

Concretely it is OLS on the three HAR features built in `00_features.ipynb`. The
subscripts $w$, $m$ and $q$ stand for **week / month / quarter** — the short, medium
and long horizons of those features: $\text{RV}^{(w)}_{t-1}$ is just last week's RV
(`rv_w_lag1`), $\text{RV}^{(m)}_{t-1}$ is the 4-week trailing mean of past RV
(`rv_m_lag1`, ≈ 1 month) and $\text{RV}^{(q)}_{t-1}$ is the 12-week trailing mean
(`rv_q_lag1`, ≈ 1 quarter). All three are `.shift(1)`-ed in `00_features.ipynb` so only
information available at the close of week $t-1$ enters the week-$t$ forecast.

$$\text{RV}_t = \beta_0 + \beta_w \text{RV}^{(w)}_{t-1} + \beta_m \text{RV}^{(m)}_{t-1} + \beta_q \text{RV}^{(q)}_{t-1} + \varepsilon_t$$

Despite its simplicity HAR-RV is the standard volatility benchmark in the literature,
because those three lags capture the long-memory persistence of volatility well.
Because OLS has no hyperparameters to tune, the validation split serves no
model-selection role here — train and val are pooled for the final fit and the model
is scored once on the held-out test set. (The RF and XGB notebooks in `03`/`04` keep
`val_df` separate because they grid-search hyperparameters on it.)

In [10]:
X_tr = sm.add_constant(trval_df[FEATS_HAR])
y_tr = trval_df['target']
X_te = sm.add_constant(test_df[FEATS_HAR])

har_fit = sm.OLS(y_tr, X_tr).fit()
print(har_fit.summary().tables[1])

har_pred = har_fit.predict(X_te).values
results.append(vol_evaluate('HAR-RV', y_test, har_pred, prev_test))


                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0085      0.003      3.196      0.002       0.003       0.014
rv_w_lag1      0.0668      0.062      1.074      0.284      -0.056       0.189
rv_m_lag1      0.3971      0.126      3.155      0.002       0.150       0.644
rv_q_lag1      0.2799      0.129      2.173      0.030       0.027       0.533
HAR-RV                          RMSE=0.03153  MAE=0.01576  R2=+0.313  DCA=0.714


## 3. DM vs the Naïve floor — does HAR-RV genuinely beat $\text{RV}_{t-1}$?

The headline RMSE table above puts Naïve and HAR-RV side by side, but it does *not* say
whether the gap is statistically meaningful. This section runs the chapter's central
test — Diebold-Mariano (1995) with Newey-West (1987) lag-1 variance via
`vol_diebold_mariano` — so the "is weekly silver volatility predictable beyond last
week's value?" question is answered inside the model's own notebook, not only in the
cross-model `evaluation.ipynb`. Negative DM = HAR-RV has lower loss; stars mark
significance.

**QLIKE is the primary loss.** Weekly silver RV is heavy-tailed enough that under
squared-error loss a handful of extreme weeks (notably the 2026 spike) can carry the
majority of the loss differential, inflating the DM variance and killing the test's
power even when the RMSE gap is real and steady. The volatility-forecasting literature
(Patton 2011) therefore compares forecasts under **QLIKE**, a proxy-robust ratio-based
loss. Squared-error DM is kept below as a reference — the gap between the two stats is
itself the heavy-tail problem in action.


In [11]:
# Diebold-Mariano: HAR-RV vs the Naive RV_{t-1} floor.
print('QLIKE loss  --  primary test:')
vol_diebold_mariano(y_test, har_pred, prev_test, 'HAR-RV', 'Naive', loss='qlike')

print('\nSquared-error loss  --  reference:')
vol_diebold_mariano(y_test, har_pred, prev_test, 'HAR-RV', 'Naive', loss='mse');


QLIKE loss  --  primary test:
HAR-RV                       vs Naive         [qlike]  DM=-2.820  p=0.005  **    -> winner: HAR-RV

Squared-error loss  --  reference:
HAR-RV                       vs Naive         [mse  ]  DM=-1.348  p=0.178  (ns)  -> winner: tie


## 4. Sub-period breakdown

RMSE and DCA for HAR-RV split by calendar year, using the shared `PERIODS` definition
from `eval_utils`. This shows whether the model holds up across the choppy 2023, the
2024 bull start, the 2025 bull run, and 2026 YTD.


In [12]:
period_har = vol_period_metrics(y_test, har_pred, prev_test, test_df.index, PERIODS)
period_har.to_csv('../../data/processed/period_har_volatility.csv')
period_har.round(4)


,n,RMSE,MAE,DCA
Period,,,,
2023 (choppy),52,0.0148,0.0122,0.6923
2024 (bull start),52,0.0145,0.0114,0.7885
2025 (bull run),52,0.0222,0.0155,0.6923
2026 (YTD),19,0.0815,0.0381,0.6316
── Full test ──,175,0.0315,0.0158,0.7143


## 5. HAR-X ablations — what does the univariate HAR-RV miss?

HAR-RV uses only the silver RV's own past. This section asks two distinct questions
on top of that baseline, with every rung fitted as plain OLS:

1. **Cross-asset spillover** — do the six EXOG cross-asset RV lags (gold, copper, USD,
   S&P500, VIX, oil) carry next-week silver-volatility information beyond HAR's own
   history? `HAR+VIX` is the single-asset control; `HAR+EXOG` is the full linear
   spillover model — the **linear sibling** of the RF/XGB models in `03`/`04`, so the
   gap between `HAR+EXOG` (here) and `RF/XGB (HAR+EXOG)` cleanly isolates whatever
   nonlinearity the trees add.
2. **Reddit sentiment** — does **public sentiment** carry information beyond HAR (and
   beyond the linear cross-asset story)? The news→volatility channel (Engle & Ng 1993),
   with Audrino, Sigrist & Ballinari (2020) as the closest precedent. Three rungs
   separate the two sentiment mechanisms (attention vs. sentiment intensity) so any
   effect is attributable.

Every rung is fitted and scored on the **same sample** — the weeks where the Reddit
features exist (the two boundary weeks drop), so the comparison is apples-to-apples.

| Rung | Adds to HAR | Tests |
|---|---|---|
| `HAR` | — | the univariate baseline |
| `HAR+VIX` | `vix_rv_lag1` | one-cross-asset spillover |
| `HAR+EXOG` | 6 cross-asset RV lags | full **linear** cross-asset spillover |
| `HAR+Attention` | `reddit_attention_lag1` | attention mechanism |
| `HAR+SentIntensity` | `reddit_sent_abs_lag1`, `reddit_sent_disp_lag1` | sentiment-intensity mechanism |
| `HAR+Attention+SentIntensity` | all three Reddit features | sentiment combined |

Significance is read from **QLIKE-DM** (Patton 2011) vs the `HAR` baseline —
squared-error DM is near-powerless on this heavy-tailed target (see `evaluation.ipynb`).
The combined sentiment rung is additionally DM-tested against **`HAR+EXOG`** (the
strongest non-sentiment baseline), so any sentiment effect must survive the full linear
cross-asset control, not just bare HAR.

TODO: explain briefly how the HAR formula changes when we add these ablations

In [ ]:
FEATS_SENT_ATTN = ['reddit_attention_lag1'] #TODO: I want to see the formula of this here 
FEATS_SENT_INT  = ['reddit_sent_abs_lag1', 'reddit_sent_disp_lag1'] #TODO: I want to see the formula of these here 
FEATS_EXOG      = [c for c in frame.columns
                   if c.endswith('_rv_lag1') and not c.startswith('silver')]

LADDER = {
    'HAR':                          FEATS_HAR,
    'HAR+VIX':                      FEATS_HAR + ['vix_rv_lag1'],
    'HAR+EXOG':                     FEATS_HAR + FEATS_EXOG,
    'HAR+Attention':                FEATS_HAR + FEATS_SENT_ATTN,
    'HAR+SentIntensity':            FEATS_HAR + FEATS_SENT_INT,
    'HAR+Attention+SentIntensity':  FEATS_HAR + FEATS_SENT_ATTN + FEATS_SENT_INT,
}

# common sample: every rung fitted/scored on the weeks where sentiment exists
sent_cols = FEATS_SENT_ATTN + FEATS_SENT_INT
abl       = frame.dropna(subset=sent_cols)
abl_trval = abl[abl['split'] != 'test']
abl_test  = abl[abl['split'] == 'test']
y_abl, prev_abl = abl_test['target'].values, abl_test['rv_w_lag1'].values
print(f'ablation sample: train+val={len(abl_trval)}  test={len(abl_test)}  '
      f'(headline HAR used {len(trval_df)}/{len(test_df)})\n')

abl_pred, abl_results = {}, []
for name, feats in LADDER.items():
    fit  = sm.OLS(abl_trval['target'], sm.add_constant(abl_trval[feats])).fit()
    pred = fit.predict(sm.add_constant(abl_test[feats])).values
    abl_pred[name] = pred
    abl_results.append(vol_evaluate(name, y_abl, pred, prev_abl))

# QLIKE-DM: each rung vs the HAR baseline (negative DM = rung better)
print()
dm = {}
for name in LADDER:
    if name == 'HAR':
        continue
    dm[name] = vol_diebold_mariano(y_abl, abl_pred[name], abl_pred['HAR'],
                                   name, 'HAR', loss='qlike')
# does the full sentiment rung beat the strongest non-sentiment baseline (HAR+EXOG)?
print()
vol_diebold_mariano(y_abl, abl_pred['HAR+Attention+SentIntensity'], abl_pred['HAR+EXOG'],
                    'HAR+Attention+SentIntensity', 'HAR+EXOG', loss='qlike')

abl_df = pd.DataFrame(abl_results)
abl_df['dm_qlike']   = abl_df['model'].map(lambda m: dm[m]['dm'] if m in dm else np.nan)
abl_df['dm_qlike_p'] = abl_df['model'].map(lambda m: dm[m]['p']  if m in dm else np.nan)
abl_df.to_csv('../../data/processed/metrics_har_sentiment_volatility.csv', index=False)
print('\nSaved metrics_har_sentiment_volatility.csv')
abl_df.round(5)


ablation sample: train+val=404  test=174  (headline HAR used 405/175)

HAR                             RMSE=0.03151  MAE=0.01564  R2=+0.316  DCA=0.718
HAR+VIX                         RMSE=0.03159  MAE=0.01608  R2=+0.313  DCA=0.684
HAR+EXOG                        RMSE=0.03290  MAE=0.01634  R2=+0.255  DCA=0.713
HAR+Attention                   RMSE=0.03143  MAE=0.01561  R2=+0.320  DCA=0.713
HAR+SentIntensity               RMSE=0.03146  MAE=0.01569  R2=+0.318  DCA=0.713
HAR+Attention+SentIntensity     RMSE=0.03146  MAE=0.01579  R2=+0.319  DCA=0.713

HAR+VIX                      vs HAR           [qlike]  DM=+1.030  p=0.303  (ns)  -> winner: tie
HAR+EXOG                     vs HAR           [qlike]  DM=+2.333  p=0.020  *     -> winner: HAR
HAR+Attention                vs HAR           [qlike]  DM=-2.833  p=0.005  **    -> winner: HAR+Attention
HAR+SentIntensity            vs HAR           [qlike]  DM=-2.857  p=0.004  **    -> winner: HAR+SentIntensity
HAR+Attention+SentIntensity  vs HAR     

,model,rmse,mae,r2,dca,dm_qlike,dm_qlike_p
0,HAR,0.03151,0.01564,0.31615,0.71839,NaN,NaN
1,HAR+VIX,0.03159,0.01608,0.31286,0.68391,1.03002,0.30300
2,HAR+EXOG,0.03290,0.01634,0.25476,0.71264,2.33281,0.01966
3,HAR+Attention,0.03143,0.01561,0.31976,0.71264,-2.83258,0.00462
4,HAR+SentIntensity,0.03146,0.01569,0.31827,0.71264,-2.85690,0.00428
5,HAR+Attention+SentIntensity,0.03146,0.01579,0.31868,0.71264,-1.83097,0.06711


**Reading the table.** `dm_qlike` / `dm_qlike_p` give the QLIKE-DM stat and p-value of
each rung vs the `HAR` baseline (negative = the rung is better; the `HAR` row is blank).
Three questions:

1. **Does linear cross-asset spillover beat bare HAR?** Read the `HAR+VIX` and
   `HAR+EXOG` rows. If `HAR+EXOG` is significantly negative, the cross-asset RVs carry
   linear silver-volatility information beyond HAR's own history.
2. **Does sentiment beat bare HAR?** A negative, significant `dm_qlike` on
   `HAR+Attention`, `HAR+SentIntensity` or the combined rung says yes.
3. **Does sentiment survive the strongest non-sentiment control?** Read the explicit
   `HAR+Att+Int vs HAR+EXOG` DM line printed above. If sentiment only matches
   `HAR+EXOG`, it is not adding anything beyond the linear cross-asset story; only
   beating `HAR+EXOG` is a genuine incremental sentiment effect.

A null result here is still thesis-relevant: *"sentiment adds nothing once the HAR
lags — and the cross-asset RVs — are in"* is clean semi-strong-form evidence,
consistent with the returns side of the thesis. Likewise, the gap between `HAR+EXOG`
(linear) here and `RF/XGB (HAR+EXOG)` in `03`/`04` measures **only** what nonlinearity
buys on top of linear cross-asset spillover.

## 6. Save outputs

- `metrics_har_volatility.csv` — Naïve + HAR-RV headline metrics
- `pred_har_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`
- `metrics_har_sentiment_volatility.csv` — the §5 sentiment-ablation table (saved above)

In [14]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_har_volatility.csv', index=False)

pred_har = pd.DataFrame({'actual': y_test, 'prev': prev_test,
                         'naive': prev_test, 'har': har_pred}, index=test_df.index)
pred_har.to_csv('../../data/processed/pred_har_volatility.csv', index_label='Date')
print('Saved metrics_har_volatility.csv + pred_har_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_har_volatility.csv + pred_har_volatility.csv


,model,rmse,mae,r2,dca
0,Naive (RV_t-1),0.03979,0.02013,-0.09462,0.00000
1,HAR-RV,0.03153,0.01576,0.31255,0.71429
